In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

In [4]:
df=pd.read_csv(r"C:\Users\hp\Downloads\fake_bills.csv",sep=';')
df

,is_genuine,diagonal,height_left,height_right,margin_low,margin_up,length
0,True,171.81,104.86,104.95,4.52,2.89,112.83
1,True,171.46,103.36,103.66,3.77,2.99,113.09
2,True,172.69,104.48,103.50,4.40,2.94,113.16
3,True,171.36,103.91,103.94,3.62,3.01,113.51
4,True,171.73,104.28,103.46,4.04,3.48,112.54
...,...,...,...,...,...,...,...
1495,False,171.75,104.38,104.17,4.42,3.09,111.28
1496,False,172.19,104.63,104.44,5.27,3.37,110.97
1497,False,171.80,104.01,104.12,5.51,3.36,111.95
1498,False,172.06,104.28,104.06,5.17,3.46,112.25


In [6]:
# =====================================================
# PART 1: DATA CLEANING
# =====================================================
df = pd.read_csv("fake_bills.csv", sep=";")   # dataset uses ';' as separator

print("Shape before cleaning:", df.shape)
print("Missing values:\n", df.isnull().sum())

df.columns = df.columns.str.strip().str.lower()
df = df.drop_duplicates()
df = df.dropna(subset=["is_genuine"])

feature_cols = [c for c in df.columns if c != "is_genuine"]

# Fill missing values (margin_low has some) with the median
df[feature_cols] = df[feature_cols].fillna(df[feature_cols].median())

df["is_genuine"] = df["is_genuine"].astype(int)   # True/False -> 1/0

# Cap extreme values using the IQR rule (set to False to skip)
CAP_OUTLIERS = True
if CAP_OUTLIERS:
    for col in feature_cols:
        q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        iqr = q3 - q1
        df[col] = df[col].clip(lower=q1 - 1.5 * iqr, upper=q3 + 1.5 * iqr)

print("\nShape after cleaning:", df.shape)
print("Number of features:", len(feature_cols), feature_cols)
print("Class balance:\n", df["is_genuine"].value_counts())

Shape before cleaning: (1500, 7)
Missing values:
 is_genuine       0
diagonal         0
height_left      0
height_right     0
margin_low      37
margin_up        0
length           0
dtype: int64

Shape after cleaning: (1500, 7)
Number of features: 6 ['diagonal', 'height_left', 'height_right', 'margin_low', 'margin_up', 'length']
Class balance:
 is_genuine
1    1000
0     500
Name: count, dtype: int64


In [7]:
X = df.drop("is_genuine", axis=1)
y = df["is_genuine"]

# 80% train, 20% test (stratify keeps class ratio equal in both)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
model = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=1000, random_state=42)),
])

model.fit(X_train, y_train)

print("Training accuracy:", round(model.score(X_train, y_train), 4))

Training accuracy: 0.9925


In [4]:
#Evaluation
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score,
    roc_curve, ConfusionMatrixDisplay
)
# Load predictions made in File 3
results = pd.read_csv("predictions.csv")

y_true = results["actual"]
y_pred = results["predicted"]
y_prob = results["probability_genuine"]
# ---------- Main metrics ----------
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_prob)

print("=" * 40)
print("MODEL EVALUATION (Logistic Regression)")
print("=" * 40)
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC AUC  : {auc:.4f}")

# ---------- Confusion matrix breakdown ----------
cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()
specificity = tn / (tn + fp)

print("\nConfusion Matrix (rows = actual, cols = predicted):")
print(cm)
print(f"\nTrue Negatives  (fake correctly found)    : {tn}")
print(f"False Positives (fake called genuine)     : {fp}")
print(f"False Negatives (genuine called fake)     : {fn}")
print(f"True Positives  (genuine correctly found) : {tp}")
print(f"Specificity (fake detection rate)         : {specificity:.4f}")

# ---------- Full report ----------
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=["Fake", "Genuine"]))
# ---------- Chart 1: Confusion matrix ----------
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Fake", "Genuine"])
disp.plot(cmap="Blues")
plt.title("Confusion Matrix")
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.close()

# ---------- Chart 2: ROC curve ----------
fpr, tpr, _ = roc_curve(y_true, y_prob)
plt.figure()
plt.plot(fpr, tpr, label=f"Logistic Regression (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.savefig("roc_curve.png", dpi=150, bbox_inches="tight")
plt.close()

print("\nSaved: confusion_matrix.png, roc_curve.png")

FileNotFoundError: [Errno 2] No such file or directory: 'predictions.csv'